# KESARi Inbound — Auto Translation Pipeline

| Step | What happens |
|------|-------------|
| 1 | Install dependencies |
| 2 | Check configuration |
| 3 | Scrape React pages → save to `INT/ENG/` |
| 4 | Translate changed pages → all language folders |
| 5 | Verify output |
| 6 | (Optional) Force translate everything from scratch |
| 7 | Auto scheduler — runs every 24 hours |

---
**Before running:**
- Run `npm run dev` in a separate terminal (Vite dev server must be on `localhost:5173`)
- Add `MISTRAL_API_KEY=your_key` to `pipeline/.env`

## Step 1 — Install Dependencies
Run once only.

In [ ]:
import subprocess, sys

print('Installing Python packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'playwright', 'beautifulsoup4', 'lxml', 'mistralai', 'python-dotenv', 'schedule'])

print('Installing Playwright browser...')
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])

print('✔ All dependencies ready')

## Step 2 — Check Configuration

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

from dotenv import load_dotenv
load_dotenv('.env')

from config import DEV_SERVER_URL, ENG_DIR, INT_DIR, LANGUAGES, ALL_ROUTES, CHECK_INTERVAL_HOURS

print(f'Dev server URL  : {DEV_SERVER_URL}')
print(f'English folder  : {ENG_DIR}')
print(f'Output folder   : {INT_DIR}')
print(f'Languages       : {list(LANGUAGES.keys())}')
print(f'Pages to scrape : {len(ALL_ROUTES)}')
print(f'Check every     : {CHECK_INTERVAL_HOURS} hours')
print()

if not os.environ.get('MISTRAL_API_KEY'):
    print('⚠️  MISTRAL_API_KEY not set — add it to pipeline/.env')
else:
    print('✔ MISTRAL_API_KEY loaded')

## Step 3 — Scrape: Render React App → Save to INT/ENG/

⚠️ Make sure `npm run dev` is running first.

Uses headless Chromium to fully render each React route, then saves the complete HTML to `INT/ENG/`.

In [ ]:
import subprocess, sys, json, os

# Run scraper as subprocess to avoid Playwright asyncio conflict in Jupyter
result = subprocess.run(
    [sys.executable, '-c',
     'import sys; sys.path.insert(0,"."); from scraper import scrape; import json; '
     'changed = scrape(verbose=True); '
     'print("__CHANGED__" + json.dumps([(r,p) for r,p in changed]))'
    ],
    capture_output=False,
    text=True,
    cwd=os.path.abspath('.')
)

print('\n✔ Scrape complete')

## Step 4 — Translate Changed Pages

Only pages that changed in Step 3 are sent to Mistral for translation.

In [ ]:
import subprocess, sys, os

# Run full pipeline (scrape + translate) as subprocess
result = subprocess.run(
    [sys.executable, 'run.py'],
    cwd=os.path.abspath('.')
)

if result.returncode == 0:
    print('\n✔ Scrape + translation complete!')
else:
    print('\n✗ Pipeline failed — check output above')

## Step 5 — Verify Output

In [ ]:
import glob, os
from config import INT_DIR, LANGUAGES

print('INT/ folder structure:\n')
for lang in ['ENG'] + list(LANGUAGES.keys()):
    folder = os.path.join(INT_DIR, lang)
    files  = glob.glob(os.path.join(folder, '**', '*.html'), recursive=True)
    status = '✔' if files else '⚠️  empty'
    print(f'  {status}  INT/{lang}/  →  {len(files)} HTML file(s)')

## Step 6 (Optional) — Force Translate ALL Pages

Use on first run or when adding a new language. Translates everything in `INT/ENG/` regardless of change detection.

In [ ]:
# Uncomment and run to force-translate every page

# import subprocess, sys, os
# subprocess.run(
#     [sys.executable, '-c',
#      'import sys; sys.path.insert(0,"."); from translator import translate_all_pages; translate_all_pages()'],
#     cwd=os.path.abspath('.')
# )

## Step 7 — Auto Scheduler (Every 24 Hours)

Keep this cell running. It calls `run.py` as a **subprocess** every 24 hours — this avoids the Playwright asyncio conflict in Jupyter.

- If pages changed → scrapes + translates automatically
- If nothing changed → prints status and waits
- Press **■ Stop** to cancel

In [ ]:
import schedule, time, datetime, subprocess, sys, os
from config import CHECK_INTERVAL_HOURS, LANGUAGES

PIPELINE_DIR = os.path.abspath('.')

def run_pipeline():
    now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    print(f'\n[{now}] ▶ Running pipeline...')
    result = subprocess.run(
        [sys.executable, 'run.py'],
        cwd=PIPELINE_DIR
    )
    if result.returncode == 0:
        print(f'  ✔ Done! Next check in {CHECK_INTERVAL_HOURS}h')
    else:
        print(f'  ✗ Pipeline failed (exit {result.returncode})')

schedule.every(CHECK_INTERVAL_HOURS).hours.do(run_pipeline)

next_run = (datetime.datetime.now() + datetime.timedelta(hours=CHECK_INTERVAL_HOURS)).strftime('%Y-%m-%d %H:%M')
print(f'✔ Scheduler started — pipeline runs every {CHECK_INTERVAL_HOURS} hours')
print(f'  First automatic run at: {next_run}')
print(f'  Languages: {list(LANGUAGES.keys())}')
print('  (Keep this cell running. Press ■ Stop to cancel)\n')

# Run once immediately on start
run_pipeline()

while True:
    schedule.run_pending()
    time.sleep(60)

---
## Quick Reference

| Task | What to run |
|------|-------------|
| First time setup | Step 1 → Step 6 |
| Manual one-off run | Step 4 |
| Auto daily run | Step 7 (keep running) |
| Check output | Step 5 |
| Add new language | Add to `config.py` LANGUAGES → run Step 6 |

```
INT/
  ENG/           ← scraped English pages (source of truth)
    index.html
    explore/index.html
    explore/product-details/INBOUND-01/index.html
    ... (~70 pages)

  es/  fr/  de/  hi/  ja/  pt/  it/
  zh/  ar/  ko/  ml/  pl/
    ← same structure, Mistral-translated
```